In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
import pandas as pd

filepath = "/content/drive/MyDrive/esi-triage-ml-vs-transformers/pubmed-rct/PubMed_200k_RCT/train.txt"

with open(filepath, 'r') as file:
    for i, line in enumerate(file):
        if i >= 100:
            break
        print(line, end='')

###24491034
BACKGROUND	The emergence of HIV as a chronic condition means that people living with HIV are required to take more responsibility for the self-management of their condition , including making physical , emotional and social adjustments .
BACKGROUND	This paper describes the design and evaluation of Positive Outlook , an online program aiming to enhance the self-management skills of gay men living with HIV .
METHODS	This study is designed as a randomised controlled trial in which men living with HIV in Australia will be assigned to either an intervention group or usual care control group .
METHODS	The intervention group will participate in the online group program ` Positive Outlook ' .
METHODS	The program is based on self-efficacy theory and uses a self-management approach to enhance skills , confidence and abilities to manage the psychosocial issues associated with HIV in daily life .
METHODS	Participants will access the program for a minimum of 90 minutes per week over sev

In [5]:
def parse_pubmed_rct(filepath):
    """
    Parses the raw PubMed RCT text file into a list of structured dictionaries.
    Each entry will contain the abstract ID, target label, sentence text, line number, and total lines.
    """
    parsed_data = []
    with open(filepath, 'r') as file:
        lines = file.readlines()

    current_abstract = []
    abstract_id = ""

    for line in lines:
        line = line.strip()
        if line.startswith("###"):
            abstract_id = line.replace("###", "")
            current_abstract = []
        elif line == "":
            # End of an abstract section: process the lines
            total_lines = len(current_abstract)
            for i, (label, text) in enumerate(current_abstract):
                parsed_data.append({
                    'abstract_id': abstract_id,
                    'label': label,
                    'text': text,
                    'line_number': i,
                    'total_lines': total_lines
                })
        else:
            # Split label and text
            parts = line.split('\t', 1)
            if len(parts) == 2:
                current_abstract.append((parts[0], parts[1]))

    return pd.DataFrame(parsed_data)

# Let's test the parser on the first few lines or a small sample
df_parsed = parse_pubmed_rct(filepath)
display(df_parsed.head(15))
print(f"\nParsed Dataset Shape: {df_parsed.shape}")

,abstract_id,label,text,line_number,total_lines
0,24491034,BACKGROUND,The emergence of HIV as a chronic condition me...,0,11
1,24491034,BACKGROUND,This paper describes the design and evaluation...,1,11
2,24491034,METHODS,This study is designed as a randomised control...,2,11
3,24491034,METHODS,The intervention group will participate in the...,3,11
4,24491034,METHODS,The program is based on self-efficacy theory a...,4,11
5,24491034,METHODS,Participants will access the program for a min...,5,11
6,24491034,METHODS,Primary outcomes are domain specific self-effi...,6,11
7,24491034,METHODS,"Secondary outcomes include : depression , anxi...",7,11
8,24491034,METHODS,"Data collection will take place at baseline , ...",8,11
9,24491034,CONCLUSIONS,Results of the Positive Outlook study will pro...,9,11



Parsed Dataset Shape: (2211861, 5)


In [6]:
# 1. Analyze Label Distribution
print("--- Label Distribution ---")
label_counts = df_parsed['label'].value_counts(normalize=True) * 100
for label, pct in label_counts.items():
    print(f"{label:<15}: {pct:.2f}%")

# 2. Engineer Relative Position Feature
df_parsed['relative_position'] = df_parsed['line_number'] / df_parsed['total_lines']

# 3. Quick Peek at the engineered features
display(df_parsed[['label', 'text', 'relative_position']].head(12))

--- Label Distribution ---
RESULTS        : 34.64%
METHODS        : 32.67%
CONCLUSIONS    : 15.36%
BACKGROUND     : 8.89%
OBJECTIVE      : 8.44%


,label,text,relative_position
0,BACKGROUND,The emergence of HIV as a chronic condition me...,0.000000
1,BACKGROUND,This paper describes the design and evaluation...,0.090909
2,METHODS,This study is designed as a randomised control...,0.181818
3,METHODS,The intervention group will participate in the...,0.272727
4,METHODS,The program is based on self-efficacy theory a...,0.363636
5,METHODS,Participants will access the program for a min...,0.454545
6,METHODS,Primary outcomes are domain specific self-effi...,0.545455
7,METHODS,"Secondary outcomes include : depression , anxi...",0.636364
8,METHODS,"Data collection will take place at baseline , ...",0.727273
9,CONCLUSIONS,Results of the Positive Outlook study will pro...,0.818182
